# R&D Feasibility Intelligence Layer - Heuristic Engines & Kinetic Models
## MoBai - AI-assisted Functional RTD Beverage Platform

This notebook prototypes the scientific rule engines and kinetic math formulas used in the 4 core feasibility evaluation modules.

### Modules Covered:
1. **Ingredient Feasibility** - Multi-criteria scoring
2. **Stability Risk** - Colloidal chemistry models
3. **Thermal Degradation** - Arrhenius kinetics
4. **COGS Calculator** - Process costing


## Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# Constants
KG_PER_LITER = 1.0
UNITS_PER_LITER = 1000


## Module 1: Ingredient Feasibility Scoring

The feasibility score uses a weighted multi-criteria approach:

$$S_{feasibility} = w_1 \cdot S_{availability} + w_2 \cdot S_{moq} + w_3 \cdot S_{cost} + w_4 \cdot S_{regulatory}$$


In [ ]:
def calculate_availability_score(in_stock_suppliers: int, total_suppliers: int) -> float:
    """Calculate availability score based on in-stock supplier ratio."""
    if total_suppliers == 0:
        return 0.0
    return (in_stock_suppliers / total_suppliers) * 100

def calculate_moq_score(moq_kg: float, batch_size_kg: float) -> float:
    """Calculate MOQ feasibility score."""
    ratio = moq_kg / batch_size_kg
    if ratio <= 0.5:
        return 100.0
    elif ratio <= 1.0:
        return 90.0
    elif ratio <= 2.0:
        return 75.0
    elif ratio <= 5.0:
        return 50.0
    else:
        return 25.0

# Example calculation
sample_availability = calculate_availability_score(2, 3)
sample_moq = calculate_moq_score(500, 1000)
print(f"Availability Score: {sample_availability:.1f}%")
print(f"MOQ Score: {sample_moq:.1f}%")


## Module 2: Stability Risk Prediction

Stability risk is calculated using colloidal chemistry principles:

- pH-protein interaction models
- Stabilizer synergy factors
- Phase separation risk assessment


In [ ]:
def calculate_ph_risk(target_ph: float, ph_min: float, ph_max: float) -> float:
    """
    Calculate pH-related stability risk.
    
    Risk is highest when pH is outside the protein's stability zone.
    """
    if ph_min <= target_ph <= ph_max:
        # Within optimal range
        distance_from_optimal = min(
            abs(target_ph - ph_min),
            abs(target_ph - ph_max)
        )
        return max(10, 40 - distance_from_optimal * 15)
    else:
        # Outside range - critical risk
        if target_ph < ph_min:
            distance = ph_min - target_ph
        else:
            distance = target_ph - ph_max
        return min(100, 70 + distance * 20)

# Test with whey protein
whey_ph_min, whey_ph_max = 4.5, 6.5
test_ph_values = [3.5, 4.0, 4.5, 5.5, 6.5, 7.0, 8.0]
ph_risks = [calculate_ph_risk(ph, whey_ph_min, whey_ph_max) for ph in test_ph_values]

plt.figure(figsize=(10, 5))
plt.plot(test_ph_values, ph_risks, 'b-o', linewidth=2, markersize=8)
plt.axvspan(whey_ph_min, whey_ph_max, alpha=0.3, color='green', label='Stable Zone')
plt.xlabel('pH')
plt.ylabel('Risk Score')
plt.title('Whey Protein pH Stability Risk Curve')
plt.legend()
plt.grid(True)
plt.show()


## Module 3: Thermal Degradation Kinetics

Thermal degradation follows first-order kinetics:

$$L = 10^{-t/D}$$

Where:
- $L$ = fraction retained
- $t$ = processing time (minutes)
- $D$ = D-value (time at given temp to reduce by 90%)

Temperature dependence via z-value:

$$D(T) = D_{ref} \cdot 10^{(T_{ref} - T)/z}$$


In [ ]:
def calculate_retention(
    d_value_121: float,  # D-value at 121C
    z_value: float,       # z-value in C
    process_temp: float,  # Process temperature
    process_time_sec: float  # Process time in seconds
) -> float:
    """
    Calculate nutrient retention after thermal processing.
    
    Uses first-order kinetics and Arrhenius equation.
    Returns retention percentage (0-100).
    """
    # Calculate D-value at process temperature
    d_value_at_temp = d_value_121 * (10 ** ((121 - process_temp) / z_value))
    
    # Convert time to minutes
    time_minutes = process_time_sec / 60
    
    # Calculate retention using first-order kinetics
    log_retention = -time_minutes / d_value_at_temp
    retention = 10 ** log_retention
    
    return max(0, min(100, retention * 100))

# Nutrient degradation parameters
nutrient_params = {
    'Vitamin C': {'d121': 0.8, 'z': 7.5},
    'Thiamine (B1)': {'d121': 0.15, 'z': 25.0},
    'Vitamin B12': {'d121': 0.5, 'z': 25.0},
    'Niacin (B3)': {'d121': 45.0, 'z': 28.0},
}

# Compare UHT vs Retort processing
uht_temp, uht_time = 140, 4  # 140C for 4 seconds
retort_temp, retort_time = 121, 1800  # 121C for 30 minutes

results = []
for nutrient, params in nutrient_params.items():
    uht_retention = calculate_retention(params['d121'], params['z'], uht_temp, uht_time)
    retort_retention = calculate_retention(params['d121'], params['z'], retort_temp, retort_time)
    results.append({
        'Nutrient': nutrient,
        'UHT (140C, 4s)': f'{uht_retention:.1f}%',
        'Retort (121C, 30min)': f'{retort_retention:.1f}%',
    })

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))


## Module 4: COGS Calculator

Cost calculation with sensitivity analysis:

$$COGS = C_{ingredients} + C_{processing} + C_{packaging} + C_{overhead}$$


In [ ]:
def calculate_ingredient_cost(
    ingredient_cost_per_kg: float,
    percentage: float,
    bottle_size_ml: int
) -> float:
    """
    Calculate ingredient cost per bottle.
    
    Args:
        ingredient_cost_per_kg: Cost per kg
        percentage: Percentage in formulation
        bottle_size_ml: Bottle size in mL
    
    Returns:
        Cost per bottle in same units as cost_per_kg
    """
    bottle_weight_kg = bottle_size_ml / 1000
    ingredient_weight_kg = bottle_weight_kg * (percentage / 100)
    return ingredient_cost_per_kg * ingredient_weight_kg

# Example formulation
formulation = [
    {'name': 'Purified Water', 'cost_per_kg': 0.15, 'percentage': 85},
    {'name': 'Whey Protein Isolate', 'cost_per_kg': 18.50, 'percentage': 3.5},
    {'name': 'Erythritol', 'cost_per_kg': 4.20, 'percentage': 5.0},
    {'name': 'Citric Acid', 'cost_per_kg': 0.95, 'percentage': 0.3},
    {'name': 'Vitamin C', 'cost_per_kg': 12.50, 'percentage': 0.05},
    {'name': 'Xanthan Gum', 'cost_per_kg': 12.80, 'percentage': 0.15},
    {'name': 'Caffeine', 'cost_per_kg': 45.00, 'percentage': 0.03},
]

bottle_size = 330  # mL

# Calculate costs
costs = []
for ing in formulation:
    cost = calculate_ingredient_cost(ing['cost_per_kg'], ing['percentage'], bottle_size)
    costs.append({
        'Ingredient': ing['name'],
        'Cost/kg': f"${ing['cost_per_kg']:.2f}",
        '%': ing['percentage'],
        'Cost/Bottle': f"${cost:.4f}"
    })

df_costs = pd.DataFrame(costs)
print(df_costs.to_string(index=False))
print(f"\nTotal Ingredient Cost: ${sum([calculate_ingredient_cost(c['cost_per_kg'], c['percentage'], bottle_size) for c in formulation]):.4f}")


## Summary: Overall Feasibility Score

The overall feasibility score combines all 4 modules:

$$S_{overall} = 0.25 S_{ingredient} + 0.25 S_{stability} + 0.20 S_{nutrition} + 0.30 S_{cost}$$

Where stability score is inverted (lower risk = higher score).


In [ ]:
# Calculate overall feasibility score
# Example scores from each module
scores = {
    'Ingredient Feasibility': 85,
    'Stability Risk (inverted)': 75,  # 100 - 25 risk
    'Nutrient Retention': 82,
    'Cost Feasibility': 78,
}

weights = {
    'Ingredient Feasibility': 0.25,
    'Stability Risk (inverted)': 0.25,
    'Nutrient Retention': 0.20,
    'Cost Feasibility': 0.30,
}

overall_score = sum(scores[k] * weights[k] for k in scores)

print(f"Overall Feasibility Score: {overall_score:.1f}/100")
print("\nBreakdown:")
for module, score in scores.items():
    print(f"  {module}: {score:.1f} (weight: {weights[module]})")

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#2ecc71', '#3498db', '#9b59b6', '#e74c3c']
bars = ax.barh(list(scores.keys()), list(scores.values()), color=colors)
ax.axvline(x=overall_score, color='black', linestyle='--', linewidth=2, label=f'Overall: {overall_score:.1f}')
ax.set_xlim(0, 100)
ax.set_xlabel('Score')
ax.set_title('R&D Feasibility Module Scores')
ax.legend()
for bar, score in zip(bars, scores.values()):
    ax.text(score + 2, bar.get_y() + bar.get_height()/2, f'{score:.1f}', va='center')
plt.tight_layout()
plt.show()
